# Johansen CO2 Storage — Sweet Spot Analysis

This notebook identifies the **optimal injection operating envelope** (the *sweet spot*) for CO2 storage in the Johansen formation.

## Two Hard Constraints
| Constraint | Physical Meaning | Limit |
|---|---|---|
| **Pressure** | Bottom-hole pressure must never fracture the caprock | BHP < 380 bar |
| **CO2 Containment** | CO2 saturation at any monitoring well must stay below detection threshold | S_CO2 < 0.001 |

## Two-Dataset Structure
- ****: 7 runs at fixed T=50 yr, Q doubles from 0.875→56 Mt/yr — maps the pressure constraint.
- ****: 4 runs at fixed Q=0.875 Mt/yr, T varies (50,100,200,400 yr) — maps the containment constraint.

## Sentinel Well
 is a **proposed** monitoring well at the updip structural boundary. CO2 saturation here signals the plume has reached the formation limit.

## Section 0 — Setup, Parameters & Data Ingestion

In [ ]:
# ============================================================
# === PARAMETERS — change only here, never inside functions ===
# ============================================================

DATA_ROOT          = "/Users/apple/Desktop/study/programming/Matlab/Plugins/MRST-2026a/core/examples/data/Johansen/well_csvs"
P_FRACTURE         = 380.0   # bar — caprock fracture pressure
S_BREACH_THRESHOLD = 0.001   # CO2 saturation fraction — containment breach trigger
INJECTOR_WELL      = "31_05_07"         # excluded from breach detection
SBOUNDARY_WELL     = "SBoundary_test_well"   # proposed boundary sentinel
REAL_OBS_WELLS     = ["31_01_01", "31_1-3_S", "31_2-5", "31_4-3", "31_05_02", "31_07_01"]

In [ ]:
import re, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.interpolate import interp1d
from pathlib import Path

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({"figure.dpi": 130, "font.family": "DejaVu Sans",
                            "axes.spines.top": False, "axes.spines.right": False})
DATA_ROOT = Path(DATA_ROOT)
print(f"Dataset root      : {DATA_ROOT}")
print(f"P_FRACTURE        = {P_FRACTURE} bar")
print(f"S_BREACH_THRESHOLD= {S_BREACH_THRESHOLD}")
print(f"Injector well     = {INJECTOR_WELL}")

In [ ]:
def parse_summary(txt_path):
    meta = {"Q_Mt_yr": None, "T_inj_yr": None, "total_injected_Mt": None, "peak_BHP_bar": None}
    try:
        text = txt_path.read_text()
        for pat, key in [(r"Injector\s+([\d.]+)\s+Mt/yr", "Q_Mt_yr"),
                         (r"Injection end year\s*:\s*([\d.]+)", "T_inj_yr"),
                         (r"Total CO2 injected\s*:\s*([\d.]+)", "total_injected_Mt"),
                         (r"Peak injector BHP\s*:\s*([\d.]+)", "peak_BHP_bar")]:
            m = re.search(pat, text)
            if m: meta[key] = float(m.group(1))
    except Exception as e:
        print(f"  [WARN] {txt_path}: {e}")
    return meta

def load_run(folder_path):
    folder_path = Path(folder_path)
    meta = parse_summary(folder_path / "simulation_summary.txt")
    meta["folder"] = str(folder_path)
    meta["timestamp"] = folder_path.name
    wells = {}
    for csv_file in sorted(folder_path.glob("*.csv")):
        try:
            wells[csv_file.stem] = pd.read_csv(csv_file)
        except Exception as e:
            print(f"  [WARN] {csv_file}: {e}")
    return {"meta": meta, "wells": wells}

def load_all_runs(subdir):
    runs = []
    for folder in sorted((DATA_ROOT / subdir).iterdir()):
        if folder.is_dir() and not folder.name.startswith("."):
            runs.append(load_run(folder))
    return runs

print("Loading runs...")
q_runs = load_all_runs("q_vary")
t_runs = load_all_runs("t_vary")
print(f"  q_vary: {len(q_runs)} runs    t_vary: {len(t_runs)} runs")

In [ ]:
rows = []
for tag, runs in [("q_vary", q_runs), ("t_vary", t_runs)]:
    for r in runs:
        m = r["meta"]
        rows.append({"Dataset": tag, "Timestamp": m["timestamp"],
                     "Q (Mt/yr)": m["Q_Mt_yr"], "T_inj (yr)": m["T_inj_yr"],
                     "Total CO2 (Mt)": m["total_injected_Mt"], "Peak BHP (bar)": m["peak_BHP_bar"],
                     "Wells": len(r["wells"])})
df_summary = pd.DataFrame(rows).sort_values(["Dataset","Q (Mt/yr)","T_inj (yr)"]).reset_index(drop=True)
display(df_summary)

## Section 1 — Constraint Analysis: Pressure

Using the  dataset (7 runs, T=50 yr fixed), we fit a cubic spline through the (Q, peak_BHP) data points and find **Q_max** — the injection rate at which BHP = P_FRACTURE = 380 bar.

Any Q > Q_max risks fracturing the caprock and losing containment irreversibly.

In [ ]:
q_bhp = sorted([(r["meta"]["Q_Mt_yr"], r["meta"]["peak_BHP_bar"]) for r in q_runs
                if r["meta"]["Q_Mt_yr"] is not None and r["meta"]["peak_BHP_bar"] is not None])
Q_vals, BHP_vals = map(np.array, zip(*q_bhp))
print(f"{'Q (Mt/yr)':>12}  |  {'Peak BHP (bar)':>14}")
print("-" * 32)
for q, b in zip(Q_vals, BHP_vals):
    print(f"{q:12.3f}  |  {b:14.2f}")

In [ ]:
Q_fine = np.linspace(Q_vals.min(), Q_vals.max(), 2000)
bhp_interp = interp1d(Q_vals, BHP_vals, kind="cubic", fill_value="extrapolate")
BHP_fine = bhp_interp(Q_fine)

Q_max = None
for i in range(len(BHP_fine) - 1):
    if (BHP_fine[i] - P_FRACTURE) * (BHP_fine[i+1] - P_FRACTURE) <= 0:
        Q_max = float(np.interp(P_FRACTURE, sorted([BHP_fine[i], BHP_fine[i+1]]),
                                            sorted([Q_fine[i], Q_fine[i+1]])))
        break
if Q_max is None:
    Q_max = Q_vals.min() if BHP_vals.min() > P_FRACTURE else Q_vals.max()

print(f">>> Q_max (caprock fracture limit) = {Q_max:.4f} Mt/yr")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.axhspan(P_FRACTURE, BHP_fine.max() * 1.06, alpha=0.12, color="red", label="Fracture Risk Zone")
ax.axhline(P_FRACTURE, color="red", linestyle="--", lw=1.8, label=f"P_FRACTURE = {P_FRACTURE} bar")
ax.plot(Q_fine, BHP_fine, color="#2980B9", lw=2.2, label="Interpolated BHP curve")
ax.scatter(Q_vals, BHP_vals, color="#E74C3C", s=80, zorder=5, label="Simulated (Q, BHP)")
ax.axvline(Q_max, color="darkorange", linestyle=":", lw=2.0, label=f"Q_max = {Q_max:.3f} Mt/yr")
ax.annotate(f"Q_max\n{Q_max:.3f} Mt/yr", xy=(Q_max, P_FRACTURE),
            xytext=(Q_max + Q_vals.max()*0.03, P_FRACTURE + 50),
            fontsize=9, color="darkorange",
            arrowprops=dict(arrowstyle="->", color="darkorange", lw=1.4))
ax.set_xlabel("Injection Rate Q (Mt/yr)", fontsize=12)
ax.set_ylabel("Peak Bottom-Hole Pressure (bar)", fontsize=12)
ax.set_title("Section 1 — BHP vs Injection Rate: Caprock Fracture Constraint", fontsize=13, fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, alpha=0.35)
ax.set_xlim(0, Q_vals.max() * 1.08)
plt.tight_layout(); plt.show()
print(f"\nResult: Q_max = {Q_max:.4f} Mt/yr — injection above this rate risks caprock fracture.")

## Section 2 — Constraint Analysis: CO2 Containment Breach

For each well and run,  is the first timestep (in years) at which the maximum CO2 saturation across layers L6–L10 exceeds . A value of 1000 means no breach occurred in the simulation window.

Color coding: 🟢 **No breach** (1000 yr), 🟡 **Late breach** (100–999 yr), 🔴 **Early breach** (<100 yr).

In [ ]:
SAT_COLS = [f"S_CO2_L{k}" for k in range(6, 11)]

def s_max_series(df):
    if df is None or df.empty: return None, None
    cols = [c for c in SAT_COLS if c in df.columns]
    if not cols: return None, None
    return df["Time_yr"], df[cols].max(axis=1)

def find_T_breach(df):
    if df is None or df.empty or "Time_yr" not in df.columns: return 1000.0
    t, s = s_max_series(df)
    if s is None: return 1000.0
    breach = df.loc[s > S_BREACH_THRESHOLD, "Time_yr"]
    return float(breach.iloc[0]) if len(breach) > 0 else 1000.0

def breach_table(runs, well_set):
    rows = []
    for r in runs:
        m = r["meta"]
        row = {"Q (Mt/yr)": m["Q_Mt_yr"], "T_inj (yr)": m["T_inj_yr"]}
        for w in well_set:
            row[w] = find_T_breach(r["wells"].get(w))
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["Q (Mt/yr)","T_inj (yr)"]).reset_index(drop=True)

def color_breach(val):
    try: v = float(val)
    except: return ""
    if v >= 1000: return "background-color:#2ECC71;color:black"
    if v >= 100:  return "background-color:#F39C12;color:black"
    return                "background-color:#E74C3C;color:white"

def style_bt(df):
    wcols = [c for c in df.columns if c not in ["Q (Mt/yr)","T_inj (yr)"]]
    return df.style.map(color_breach, subset=wcols).format({c:"{:.0f}" for c in wcols})

ALL_RUNS     = q_runs + t_runs
MONITOR_REAL = REAL_OBS_WELLS
MONITOR_ALL  = REAL_OBS_WELLS + [SBOUNDARY_WELL]
print("Functions defined.")

In [ ]:
def plot_saturation_panels(runs, well_set, prefix=""):
    n = len(well_set)
    for r in runs:
        m = r["meta"]
        lbl = f"Q={m['Q_Mt_yr']} Mt/yr, T_inj={m['T_inj_yr']} yr"
        fig, axes = plt.subplots(1, n, figsize=(3.2*n, 3.0), sharey=False)
        if n == 1: axes = [axes]
        fig.suptitle(f"{prefix}  |  {lbl}", fontsize=9, fontweight="bold")
        for ax, w in zip(axes, well_set):
            df = r["wells"].get(w)
            tb = find_T_breach(df)
            if df is not None and not df.empty and "Time_yr" in df.columns:
                t, s = s_max_series(df)
                if s is not None: ax.plot(t, s, lw=1.4, color="#2980B9")
            ax.axhline(S_BREACH_THRESHOLD, color="red", ls="--", lw=1.2, alpha=0.8)
            col = "#E74C3C" if tb < 1000 else "#27AE60"
            ax.set_title(f"{w}\nT_b={tb:.0f}yr", fontsize=7, color=col)
            ax.set_xlabel("Time (yr)", fontsize=7); ax.set_ylabel("S_CO2 max", fontsize=7)
            ax.tick_params(labelsize=6); ax.grid(True, alpha=0.25)
        plt.tight_layout(); plt.show()

print("--- q_vary saturation profiles (real wells) ---")
plot_saturation_panels(q_runs, MONITOR_REAL, "q_vary | Real Wells")

In [ ]:
print("--- t_vary saturation profiles (real wells) ---")
plot_saturation_panels(t_runs, MONITOR_REAL, "t_vary | Real Wells")

In [ ]:
print("BREACH TABLE — q_vary — REAL WELLS ONLY")
display(style_bt(breach_table(q_runs, MONITOR_REAL)))
print("\nBREACH TABLE — q_vary — WITH SBoundary")
display(style_bt(breach_table(q_runs, MONITOR_ALL)))
print("\nBREACH TABLE — ALL RUNS — REAL WELLS ONLY")
display(style_bt(breach_table(ALL_RUNS, MONITOR_REAL)))
print("\nBREACH TABLE — ALL RUNS — WITH SBoundary")
display(style_bt(breach_table(ALL_RUNS, MONITOR_ALL)))

## Section 3 — Sweet Spot Region: Real Wells Only

We combine both constraints into a Q–T feasibility map.

- **T_safe(Q)** = earliest T_breach across all real observation wells for that Q.
- **Feasible region**: Q ≤ Q_max AND T_inj ≤ T_safe(Q)
- **Optimal point**: the (Q, T) pair on the boundary that maximises V = Q × T_inj (Mt stored)

Iso-volume contours (grey dashed) show how much CO2 is stored for each Q–T combination.

In [ ]:
def compute_T_safe(runs, well_set):
    result = []
    for r in runs:
        Q = r["meta"]["Q_Mt_yr"]
        if Q is None: continue
        breaches = [find_T_breach(r["wells"].get(w)) for w in well_set]
        result.append((Q, min(breaches) if breaches else 1000.0))
    return sorted(result)

def sweet_spot_plot(q_safe_pairs, Q_max_val, title_suffix, color="#2ECC71", ax=None):
    show = ax is None
    if ax is None: fig, ax = plt.subplots(figsize=(9, 6))
    Qs = np.array([p[0] for p in q_safe_pairs])
    Ts = np.array([p[1] for p in q_safe_pairs])
    valid = Qs <= Q_max_val
    Qs_f, Ts_f = Qs[valid], Ts[valid]
    if len(Qs_f) >= 2:
        Q_dense = np.linspace(Qs_f.min(), Qs_f.max(), 400)
        kind = "linear" if len(Qs_f) < 4 else "cubic"
        fn = interp1d(Qs_f, Ts_f, kind=kind, fill_value="extrapolate")
        T_dense = np.clip(fn(Q_dense), 0, 1000)
    else:
        Q_dense, T_dense = Qs_f, Ts_f
    T_max_p = min(float(Ts.max()) * 1.15, 1100)
    ax.fill_between(Q_dense, 0, T_dense, alpha=0.25, color=color, label="Feasible region")
    ax.plot(Q_dense, T_dense, color=color, lw=2.2, label="T_safe boundary")
    ax.scatter(Qs_f, Ts_f, color=color, s=70, zorder=5)
    ax.axvline(Q_max_val, color="red", ls="--", lw=1.8, label=f"Q_max={Q_max_val:.3f} Mt/yr")
    # Iso-volume contours
    Qc = np.linspace(0.01, Q_max_val * 1.3, 300)
    for V in np.geomspace(10, Q_max_val * T_max_p * 0.85, 8):
        Ti = V / Qc; mask = Ti <= T_max_p * 1.05
        if mask.any():
            ax.plot(Qc[mask], Ti[mask], "grey", ls="--", lw=0.8, alpha=0.55)
            li = np.where(mask)[0][-1]
            ax.text(Qc[li], Ti[li], f"{V:.0f} Mt", fontsize=6, color="grey", alpha=0.75)
    # Optimal
    Vb = Q_dense * T_dense; oi = np.argmax(Vb)
    Q_opt, T_opt, V_opt = Q_dense[oi], T_dense[oi], Vb[oi]
    ax.scatter(Q_opt, T_opt, marker="*", s=320, color="gold", edgecolors="darkorange",
               lw=1.5, zorder=10, label=f"Optimal: V={V_opt:.1f} Mt")
    ax.annotate(f" Q={Q_opt:.2f}\n T={T_opt:.0f}yr\n V={V_opt:.1f}Mt",
                xy=(Q_opt, T_opt), xytext=(Q_opt+Q_max_val*0.04, T_opt+T_max_p*0.04),
                fontsize=8, color="darkorange",
                arrowprops=dict(arrowstyle="->", color="darkorange", lw=1.2))
    ax.set_xlabel("Injection Rate Q (Mt/yr)", fontsize=12)
    ax.set_ylabel("Injection Duration T_inj (years)", fontsize=12)
    ax.set_title(f"Sweet Spot Region — {title_suffix}", fontsize=13, fontweight="bold")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.set_xlim(0, Q_max_val * 1.3); ax.set_ylim(0, T_max_p)
    if show: plt.tight_layout(); plt.show()
    return Q_opt, T_opt, V_opt, Q_dense, T_dense

print("T_safe values (real wells):")
q_safe_real = compute_T_safe(q_runs, MONITOR_REAL)
for Q, Ts in q_safe_real:
    print(f"  Q={Q:8.3f} Mt/yr  ->  T_safe={Ts:.0f} yr")

In [ ]:
Q_opt, T_opt, V_max, Q_dense_real, T_dense_real = sweet_spot_plot(
    q_safe_real, Q_max, "Real Wells Only", color="#2ECC71")
print(f"\n>>> Q_opt={Q_opt:.4f} Mt/yr   T_opt={T_opt:.1f} yr   V_max={V_max:.2f} Mt")

## Section 4 — Sweet Spot Region: With SBoundary_test_well

We repeat the analysis including the proposed boundary sentinel. Because  sits at the updip structural limit, it may give an earlier  than interior wells — tightening the safe injection window.

Comparing with Section 3 reveals whether the sentinel well reveals a hidden escape pathway or simply confirms the safe envelope.

In [ ]:
print("T_safe values (with SBoundary):")
q_safe_sb = compute_T_safe(q_runs, MONITOR_ALL)
for Q, Ts in q_safe_sb:
    print(f"  Q={Q:8.3f} Mt/yr  ->  T_safe={Ts:.0f} yr")

In [ ]:
Q_opt_sb, T_opt_sb, V_max_sb, Q_dense_sb, T_dense_sb = sweet_spot_plot(
    q_safe_sb, Q_max, "With SBoundary_test_well", color="#2980B9")
print(f"\n>>> Q_opt_sb={Q_opt_sb:.4f} Mt/yr   T_opt_sb={T_opt_sb:.1f} yr   V_max_sb={V_max_sb:.2f} Mt")

## Section 5 — Comparative Analysis: Value of the SBoundary Well

Three panels answer the key question: **should we drill and monitor the ?**

- **Panel 1**: Overlay of both feasibility envelopes — shows whether the sentinel shrinks or confirms the safe region.
- **Panel 2**: Bar chart comparing maximum storable volumes — quantifies the risk or gain.
- **Panel 3**: Side-by-side T_breach at Q=0.875 Mt/yr — shows physical intuition for plume reach.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Section 5 — Comparative Analysis: Value of the SBoundary Sentinel Well",
             fontsize=13, fontweight="bold")

# ── Panel 1: Overlaid feasibility regions ────────────────────────────────────
ax1 = axes[0]
T_mp = 1100
ax1.fill_between(Q_dense_real, 0, T_dense_real, alpha=0.30, color="#2ECC71",
                 label="Feasible (real wells)")
ax1.plot(Q_dense_real, T_dense_real, color="#27AE60", lw=2.2)
ax1.fill_between(Q_dense_sb, 0, T_dense_sb, alpha=0.22, color="#2980B9",
                 hatch="//", label="Feasible (+ SBoundary)")
ax1.plot(Q_dense_sb, T_dense_sb, color="#2980B9", lw=2.0, ls="-.")
ax1.scatter(Q_opt, T_opt, marker="*", s=280, color="gold", edgecolors="darkorange",
            lw=1.5, zorder=10, label=f"Opt (real): V={V_max:.1f} Mt")
ax1.scatter(Q_opt_sb, T_opt_sb, marker="*", s=280, color="#2980B9", edgecolors="navy",
            lw=1.5, zorder=10, label=f"Opt (SB): V={V_max_sb:.1f} Mt")
ax1.axvline(Q_max, color="red", ls="--", lw=1.5, label=f"Q_max={Q_max:.2f}")
ax1.set_xlabel("Q (Mt/yr)", fontsize=11); ax1.set_ylabel("T_inj (years)", fontsize=11)
ax1.set_title("Panel 1: Overlay of Sweet Spot Regions", fontsize=11, fontweight="bold")
ax1.legend(fontsize=7.5, loc="upper right"); ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, Q_max * 1.3); ax1.set_ylim(0, T_mp)

# ── Panel 2: Volume bar chart ─────────────────────────────────────────────────
ax2 = axes[1]
bars = ax2.bar(["Without\nSBoundary","With\nSBoundary"], [V_max, V_max_sb],
               color=["#2ECC71","#2980B9"], edgecolor="white", width=0.5)
for bar, val in zip(bars, [V_max, V_max_sb]):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+V_max*0.01,
             f"{val:.1f} Mt", ha="center", va="bottom", fontsize=12, fontweight="bold")
delta = V_max_sb - V_max
msg = (f"Containment Risk Revealed:\n{abs(delta):.1f} Mt avoided" if delta < -0.5 else
       f"Confirmed safe volume:\n{delta:.1f} Mt additional" if delta > 0.5 else
       f"Equivalent safe volumes\n(SBoundary confirms envelope)")
col = "#E74C3C" if delta < -0.5 else "#27AE60"
ax2.text(0.5, 0.58, msg, ha="center", va="center", transform=ax2.transAxes,
         fontsize=10, color=col, fontweight="bold",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor=col, alpha=0.85))
ax2.set_ylabel("Max Storable Volume (Mt CO2)", fontsize=10)
ax2.set_title("Panel 2: Volume Gain/Loss from\nSBoundary Sentinel", fontsize=11, fontweight="bold")
ax2.grid(True, alpha=0.3, axis="y"); ax2.set_ylim(0, max(V_max, V_max_sb) * 1.28)

# ── Panel 3: T_breach at Q=0.875 ─────────────────────────────────────────────
ax3 = axes[2]
q_target = 0.875
base_run = min(q_runs, key=lambda r: abs((r["meta"]["Q_Mt_yr"] or 1e9) - q_target))
tb_real = [find_T_breach(base_run["wells"].get(w)) for w in MONITOR_REAL]
tb_sb   = find_T_breach(base_run["wells"].get(SBOUNDARY_WELL))
all_w = MONITOR_REAL + [SBOUNDARY_WELL]
all_tb = tb_real + [tb_sb]
all_col = ["#1ABC9C"]*len(MONITOR_REAL) + ["#E67E22"]
lbls = [w if len(w)<=8 else w.replace("_","\n") for w in all_w]
lbls[-1] = "SBoundary"

b3 = ax3.bar(range(len(all_w)), all_tb, color=all_col, edgecolor="white")
ax3.set_xticks(range(len(all_w))); ax3.set_xticklabels(lbls, fontsize=7, rotation=30, ha="right")
ax3.axhline(1000, color="green", ls=":", lw=1.5, alpha=0.7, label="No breach in 1000 yr")
for bar, val in zip(b3, all_tb):
    ax3.text(bar.get_x()+bar.get_width()/2, min(val+25, 975),
             f"{val:.0f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
ax3.legend(handles=[mpatches.Patch(color="#1ABC9C",label="Real obs. wells"),
                    mpatches.Patch(color="#E67E22",label="SBoundary")], fontsize=8)
ax3.set_ylabel("T_breach (years)", fontsize=10)
ax3.set_title(f"Panel 3: Breach Timeline at Q={q_target} Mt/yr", fontsize=11, fontweight="bold")
ax3.set_ylim(0, 1150); ax3.grid(True, alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

In [ ]:
delta_V = V_max_sb - V_max
if delta_V < -0.5:
    verdict = (f"The SBoundary sentinel well reveals a CO2 escape pathway invisible to the interior "
                f"monitoring network. Without it, operators might store {V_max:.1f} Mt but risk "
                f"{abs(delta_V):.1f} Mt of undetected containment failure. "
                f"The sentinel well is CRITICAL for regulatory compliance.")
elif abs(delta_V) <= 0.5:
    verdict = (f"The SBoundary sentinel well confirms the safe operating envelope. The interior "
                f"network already captures the plume boundary (optimal volume = {V_max:.1f} Mt). "
                f"The sentinel provides valuable monitoring redundancy but is not strictly required.")
else:
    verdict = (f"The SBoundary sentinel well reveals additional formation capacity beyond what "
                f"interior wells suggest, allowing {delta_V:.1f} Mt extra storage. "
                f"It is VALUABLE for unlocking maximum reservoir potential.")

box = "=" * 78
print(f"""
{box}
  JOHANSEN CO2 STORAGE — SWEET SPOT OPTIMISATION SUMMARY
{box}
  PRESSURE CONSTRAINT
    Fracture pressure : {P_FRACTURE} bar
    Q_max             : {Q_max:.4f} Mt/yr

  OPTIMAL POINT — Real Wells Only
    Q_opt = {Q_opt:.4f} Mt/yr   |   T_opt = {T_opt:.1f} yr   |   V_max = {V_max:.2f} Mt

  OPTIMAL POINT — With SBoundary Sentinel
    Q_opt = {Q_opt_sb:.4f} Mt/yr   |   T_opt = {T_opt_sb:.1f} yr   |   V_max = {V_max_sb:.2f} Mt
{box}
""")
print("SENTINEL WELL VERDICT:")
print(verdict)